# Processing Audio File Tags

In [1]:
%load_ext autoreload
%autoreload 2

In [12]:
from pathlib import Path
from aft.tags import read_audio_tags
from aft.ingest import detect_artist_album_from_path, normalize_filename
from aft.utils.normalization import normalize_spaces

from aft.credentials import discogs_user_token
from aft.discogs_client import DiscogsClient

In [13]:
# file_path = Path(r"D:\Soulseek Downloads\complete\Dragutesku, Search DiP - Prelude [DRGL002]\Dragutesku, Search DiP - Prelude (Original Mix).mp3")
file_path = Path(r"D:\Music Collection\214\Esemplastic\01 twofourteen - her stereo malanger.mp3")
source_dir = Path(r"D:\Soulseek Downloads\complete")
dest_root = Path(r"D:\Music Collection")

normalized_filename = normalize_filename(file_path.name)
print(f"Normalized Filename: {normalized_filename}")

Normalized Filename: 01 twofourteen - her stereo malanger.mp3


## Get Artist, Album and Title

### From tags

In [26]:
tags = read_audio_tags(file_path)
display(tags)

{'title': 'Her Stereo Malanger',
 'artist': '214',
 'album': 'Esemplastic',
 'album_artist': '214',
 'year': '2004',
 'genre': 'IDM; Electro; Ambient',
 'track_number': '1/11',
 'disc_number': '1/1',
 'bpm': '',
 'key': '',
 'publisher': 'Mikrolux',
 'catalog_number': '',
 'comment': 'MKX11CD'}

In [5]:
# Try to get artist/album from tags or path
artist = tags.get('artist') or ''
album = tags.get('album') or ''
title = tags.get('title') or file_path.stem
existing_bpm = tags.get('bpm')

# Normalize names
artist = normalize_spaces(artist)
album = normalize_spaces(album)
title = normalize_spaces(title)

print(f"Artist: {artist}")
print(f"Album: {album}")
print(f"Title: {title}")
print(f"BPM: {existing_bpm}")

Artist: Dragutesku, Search DiP
Album: www.electronicfresh.com
Title: Prelude (Original Mix)
BPM: 126


### From file path

In [6]:
detected_artist, detected_album = detect_artist_album_from_path(
    file_path,
    source_dir
)

print(f"Detected Artist: {detected_artist}")
print(f"Detected Album: {detected_album}")

Detected Artist: Dragutesku, Search DiP
Detected Album: Prelude [DRGL002]


In [7]:
# Build destination path: dest_root / artist / album / filename
normalized_filename = normalize_filename(file_path.name)
dest_dir = dest_root / artist / album
dest_path = dest_dir / normalized_filename

print(f"Destination Path: {dest_path}")

Destination Path: D:\Music Collection\Dragutesku, Search DiP\www.electronicfresh.com\Dragutesku, Search DiP - Prelude (Original Mix).mp3


## Search metadata using Discogs API

In [9]:
# Instantiate the Discogs client with the user token
if discogs_user_token is None:
	raise ValueError("Discogs user token is not set. Please provide a valid user token.")
discogs_client = DiscogsClient(user_token=discogs_user_token)

### Artist information

In [18]:
artist_name = "Dragutesku"
results = discogs_client.search_artist(artist_name=artist_name)
print(f"Discogs Search Results for '{artist_name}':")
for result in results:
    print(f"- {result['name']} (ID: {result['id']})")

Discogs Search Results for 'Dragutesku':
- Dragutesku (ID: 4891337)


In [27]:
artist_name = "Search DiP"
results = discogs_client.search_artist(artist_name=artist_name)
print(f"Discogs Search Results for '{artist_name}':")
for result in results:
    print(f"- {result['name']} (ID: {result['id']})")

Discogs Search Results for 'Search DiP':
- Search DiP (ID: 2575814)


In [26]:
selected_artist_id = 4891337
results = discogs_client.get_artist_by_id(artist_id=selected_artist_id)
if results is not None:
    print(f"Artist Information for ID '{selected_artist_id}':")
    print(f"Name: {results['name']}")
    print(f"Real Name: {results['real_name']}")
    print(f"Profile: {results['profile']}")
    print(f"Members: {results['members']}")
    print(f"Aliases: {results['aliases']}")
    print(f"Name Variations: {results['name_variations']}")
    print(f"URLs: {results['urls']}")
    print(f"Images: {results['images']}")

Artist Information for ID '4891337':
Name: Dragutesku
Real Name: Dragut Adrian
Profile: Bucharest, Romania.
Members: []
Aliases: ['Dragut Adrian']
Name Variations: None
URLs: ['https://soundcloud.com/dragutesku', 'https://www.facebook.com/dragutesku/', 'https://www.instagram.com/dragutesku/', 'https://dragutesku.bandcamp.com', 'https://www.residentadvisor.net/dj/dragutesku']
Images: [{'type': 'primary', 'uri': 'https://i.discogs.com/U6I3Ab-ydL4PC6IWEj3z7HLGtU0CvanQoT8eots1a6g/rs:fit/g:sm/q:90/h:600/w:600/czM6Ly9kaXNjb2dz/LWRhdGFiYXNlLWlt/YWdlcy9BLTQ4OTEz/MzctMTU4OTU4MDQ3/MC0yNzcyLmpwZWc.jpeg', 'resource_url': 'https://i.discogs.com/U6I3Ab-ydL4PC6IWEj3z7HLGtU0CvanQoT8eots1a6g/rs:fit/g:sm/q:90/h:600/w:600/czM6Ly9kaXNjb2dz/LWRhdGFiYXNlLWlt/YWdlcy9BLTQ4OTEz/MzctMTU4OTU4MDQ3/MC0yNzcyLmpwZWc.jpeg', 'uri150': 'https://i.discogs.com/QKcNyOGbJDKxhkooTGYlJ2NlLB1UfKXNct3272SLi5s/rs:fit/g:sm/q:40/h:150/w:150/czM6Ly9kaXNjb2dz/LWRhdGFiYXNlLWlt/YWdlcy9BLTQ4OTEz/MzctMTU4OTU4MDQ3/MC0yNzcyLmpwZWc.jpeg',

In [28]:
selected_artist_id = 2575814
results = discogs_client.get_artist_by_id(artist_id=selected_artist_id)
if results is not None:
    print(f"Artist Information for ID '{selected_artist_id}':")
    print(f"Name: {results['name']}")
    print(f"Real Name: {results['real_name']}")
    print(f"Profile: {results['profile']}")
    print(f"Members: {results['members']}")
    print(f"Aliases: {results['aliases']}")
    print(f"Name Variations: {results['name_variations']}")
    print(f"URLs: {results['urls']}")
    print(f"Images: {results['images']}")

Artist Information for ID '2575814':
Name: Search DiP
Real Name: Mihai Leonte
Profile: DJ and producer from Bucharest, Romania.
Members: []
Aliases: ['Mihai Leonte']
Name Variations: None
URLs: ['https://www.facebook.com/searchdip', 'https://soundcloud.com/searchdip', 'https://www.instagram.com/searchdip/']
Images: [{'type': 'primary', 'uri': 'https://i.discogs.com/E5AAnCj4PoP1LuvX8KEpqLIQAuUulouu1yHIVp0aMM0/rs:fit/g:sm/q:90/h:546/w:546/czM6Ly9kaXNjb2dz/LWRhdGFiYXNlLWlt/YWdlcy9BLTI1NzU4/MTQtMTU2MTczNDkx/MC04MTA5LmpwZWc.jpeg', 'resource_url': 'https://i.discogs.com/E5AAnCj4PoP1LuvX8KEpqLIQAuUulouu1yHIVp0aMM0/rs:fit/g:sm/q:90/h:546/w:546/czM6Ly9kaXNjb2dz/LWRhdGFiYXNlLWlt/YWdlcy9BLTI1NzU4/MTQtMTU2MTczNDkx/MC04MTA5LmpwZWc.jpeg', 'uri150': 'https://i.discogs.com/zdOnjGUmOLVtokkapx4aOoX8TDeCeRf9fh6OKVb2BmI/rs:fit/g:sm/q:40/h:150/w:150/czM6Ly9kaXNjb2dz/LWRhdGFiYXNlLWlt/YWdlcy9BLTI1NzU4/MTQtMTU2MTczNDkx/MC04MTA5LmpwZWc.jpeg', 'width': 546, 'height': 546}]


### Release information

In [33]:
artist = "Dragutesku"
release = "Prelude"
results = discogs_client.search_release(release_title=release, artist=artist)
print(f"Discogs Release Search Results for '{artist} - {release}':")
for result in results:
    print(f"- {result['title']} (ID: {result['id']})")
    print(f"Artist: {result['artist']}")
    print(f"Year: {result['year']}")
    print(f"Format: {result['format']}")
    print(f"Country: {result['country']}")

Discogs Release Search Results for 'Dragutesku - Prelude':
- Prelude (ID: 27449376)
Artist: ['Dragutesku', 'Search DiP']
Year: 2023
Format: ['Vinyl']
Country: Romania


In [34]:
release_id = 27449376
release_info = discogs_client.get_release_by_id(release_id=release_id)
if release_info is not None:
    print(f"Release Information for ID '{release_id}':")
    print(f"Title: {release_info['title']}")
    print(f"Artist: {release_info['artist']}")
    print(f"Year: {release_info['year']}")
    print(f"Generes: {release_info['genres']}")
    print(f"Styles: {release_info['styles']}")
    print(f"Labels: {release_info['labels']}")
    print(f"Format: {release_info['format']}")
    print(f"Catalog Numbers: {release_info['catalog_numbers']}")
    print(f"Country: {release_info['country']}")
    print(f"Notes: {release_info['notes']}")
    print(f"Images: {release_info['images']}")
    print(f"Tracklist: {release_info['tracklist']}")

Release Information for ID '27449376':
Title: Prelude
Artist: ['Dragutesku', 'Search DiP']
Year: 2023
Generes: ['Electronic']
Styles: ['House', 'Minimal', 'Tech House']
Labels: ['DRG LIMITED']
Format: ['Vinyl']
Catalog Numbers: ['DRGL002']
Country: Romania
Notes: None
Images: [{'type': 'secondary', 'uri': 'https://i.discogs.com/dBTEOUxWSAMF3IKV3f1x2P6fqoRgFobogtqUsV9VJhI/rs:fit/g:sm/q:90/h:600/w:600/czM6Ly9kaXNjb2dz/LWRhdGFiYXNlLWlt/YWdlcy9SLTI3NDQ5/Mzc2LTE2ODczNTg5/MTctNjI3NS5qcGVn.jpeg', 'resource_url': 'https://i.discogs.com/dBTEOUxWSAMF3IKV3f1x2P6fqoRgFobogtqUsV9VJhI/rs:fit/g:sm/q:90/h:600/w:600/czM6Ly9kaXNjb2dz/LWRhdGFiYXNlLWlt/YWdlcy9SLTI3NDQ5/Mzc2LTE2ODczNTg5/MTctNjI3NS5qcGVn.jpeg', 'uri150': 'https://i.discogs.com/HA_3r4FtiTwNSHAqwETK5dfpbsJKKG3gU3Sk8dYrqVY/rs:fit/g:sm/q:40/h:150/w:150/czM6Ly9kaXNjb2dz/LWRhdGFiYXNlLWlt/YWdlcy9SLTI3NDQ5/Mzc2LTE2ODczNTg5/MTctNjI3NS5qcGVn.jpeg', 'width': 600, 'height': 600}, {'type': 'secondary', 'uri': 'https://i.discogs.com/1rF14CKfD-06c09uVfuiX

In [ ]:
from typing import List, Dict, Any, Tuple, Optional
from pathlib import Path
from mutagen.mp3 import MP3
from mutagen.id3 import ID3, TALB, TCON, TDRC, TPUB, TXXX

import shutil


## Get artist releases available in music library

In [35]:
# Set the path to the music library
download_dir = Path("D:/Soulseek Downloads/complete")

# Get directories
complete_dir = [d for d in download_dir.iterdir() if d.is_dir()]
display(complete_dir)

[WindowsPath('D:/Soulseek Downloads/complete/(2017) Analogical Force - Voiceless Y'),
 WindowsPath('D:/Soulseek Downloads/complete/(2017) Analogical Force - Voiceless Z'),
 WindowsPath('D:/Soulseek Downloads/complete/(af007) brainwaltzera - outdives ep (2017)'),
 WindowsPath('D:/Soulseek Downloads/complete/(AF012) James Shinra - Supernova EP (2018)'),
 WindowsPath('D:/Soulseek Downloads/complete/(KARAT 04) Ark - Unknow Mysterioso'),
 WindowsPath('D:/Soulseek Downloads/complete/(WOLF2BAR01) Frits Wentink \u200e– Two Bar House Music & Chord Stuff Volume One (2017) [Vinyl - FLAC]'),
 WindowsPath('D:/Soulseek Downloads/complete/0000 - Labyrinth'),
 WindowsPath('D:/Soulseek Downloads/complete/1995-Project Logic'),
 WindowsPath('D:/Soulseek Downloads/complete/2001 - Call Me Mister Falcon - Paper World'),
 WindowsPath('D:/Soulseek Downloads/complete/2002 - Gravity'),
 WindowsPath('D:/Soulseek Downloads/complete/2003 - Enemy & Lovers'),
 WindowsPath('D:/Soulseek Downloads/complete/2008 - The K

## Select tracks to tag

In [36]:
# Choose a release
idx = 4

# Get all releases directories
tracks_paths = [d for d in complete_dir[idx].iterdir() if d.is_file() and d.suffix in ['.mp3', '.flac']]
print(f"Found {len(tracks_paths)} tracks in {complete_dir[idx].name}")
display(tracks_paths)

# Get all cover art files
cover_art_paths = [d for d in complete_dir[idx].iterdir() if d.is_file() and d.suffix in ['.jpg', '.png']]
print(f"Found {len(cover_art_paths)} cover arts in {complete_dir[idx].name}")
display(cover_art_paths)

Found 5 tracks in (KARAT 04) Ark - Unknow Mysterioso


[WindowsPath('D:/Soulseek Downloads/complete/(KARAT 04) Ark - Unknow Mysterioso/A1 - Taimz.flac'),
 WindowsPath('D:/Soulseek Downloads/complete/(KARAT 04) Ark - Unknow Mysterioso/A2 - Enveloppe.flac'),
 WindowsPath('D:/Soulseek Downloads/complete/(KARAT 04) Ark - Unknow Mysterioso/A3 - Surphase.flac'),
 WindowsPath('D:/Soulseek Downloads/complete/(KARAT 04) Ark - Unknow Mysterioso/A4 - Damnark.flac'),
 WindowsPath('D:/Soulseek Downloads/complete/(KARAT 04) Ark - Unknow Mysterioso/B - Sueur De Table.flac')]

Found 2 cover arts in (KARAT 04) Ark - Unknow Mysterioso


[WindowsPath('D:/Soulseek Downloads/complete/(KARAT 04) Ark - Unknow Mysterioso/face A.png'),
 WindowsPath('D:/Soulseek Downloads/complete/(KARAT 04) Ark - Unknow Mysterioso/face B.png')]

## Search artist

In [37]:
artist = "Ark"
discogs_artists = discogs_client.search_artist(artist_name=artist)
display(discogs_artists)

[{'id': 1061, 'artist_name': 'Ark'},
 {'id': 805517, 'artist_name': 'ARK (7)'},
 {'id': 162636, 'artist_name': 'Ark (2)'},
 {'id': 7014667, 'artist_name': 'Ark (30)'},
 {'id': 1033525, 'artist_name': 'Ark (8)'},
 {'id': 84888, 'artist_name': 'The Ark'},
 {'id': 621038, 'artist_name': 'Victor Ark'},
 {'id': 117507, 'artist_name': 'T. Ark'},
 {'id': 2235329, 'artist_name': 'Danielle Van Ark'},
 {'id': 370173, 'artist_name': 'Ark (4)'}]

In [38]:
artist_id = 1061
discogs_artist_metadata = discogs_client.get_artist_by_id(artist_id=artist_id)
display(discogs_artist_metadata)

{'id': 1061,
 'artist_name': 'Ark',
 'real_name': 'Guillaume Berroyer',
 'profile': 'Son of [a=Jackie Berroyer]',
 'members': [],
 'aliases': ['Unknown Mysterioso',
  'Mr. Full Destructor',
  'Guillaume Berroyer'],
 'name_variations': ['Ark Of Light']}

## Search for release

In [39]:
release_title = "Unknow Mysterioso"
discogs_releases = discogs_client.search_release(
    release_title=release_title,
    artist=discogs_artist_metadata["artist_name"],
)

display(discogs_releases)

[{'id': 152374,
  'title': 'Ark - Unknow Mysterioso Vol. 2',
  'artist': ['Ark'],
  'year': 2000,
  'format': ['Vinyl']},
 {'id': 248089,
  'title': 'Ark - Unknow Mysterioso',
  'artist': ['Ark'],
  'year': 2000,
  'format': ['Vinyl']}]

In [40]:
release_id = 152374
discogs_release_metadata = discogs_client.get_release_by_id(release_id=release_id)
display(discogs_release_metadata)

{'id': 152374,
 'title': 'Unknow Mysterioso Vol. 2',
 'artist': ['Ark'],
 'year': 2000,
 'genres': ['Electronic'],
 'styles': ['House', 'Abstract'],
 'labels': ['Karat Records'],
 'format': ['Vinyl'],
 'catalog_numbers': ['KARAT 04'],
 'country': 'France',
 'tracklist': [{'position': 'A1', 'title': 'Taimz'},
  {'position': 'A2', 'title': 'Enveloppe'},
  {'position': 'A3', 'title': 'Surphase'},
  {'position': 'A4', 'title': 'Damnark'},
  {'position': 'B', 'title': 'Sueur De Table'}]}

## Update tracks metadata

In [41]:
# Get available metadata from tracks
def get_track_metadata(track_path: Path) -> Optional[Dict[str, Any]]:
    """
    Get metadata from an audio track file.

    Parameters
    ----------
    track_path : Path
        Path to the audio track file.
        
    Returns
    -------
    Dict[str, Any]
        Dictionary with metadata from the audio track.
    """
    try:
        audio = MP3(track_path, ID3=ID3)
        metadata = {
            'title': audio.tags.get('TIT2', 'Unknown Title'),
            'artist': audio.tags.get('TPE1', 'Unknown Artist'),
            'album': audio.tags.get('TALB', 'Unknown Album'),
            'genre': audio.tags.get('TCON', 'Unknown Genre'),
            'bpm': audio.tags.get('TBPM', 'Unknown BPM'),
            'year': audio.tags.get('TDRC', 'Unknown Year'),
            'track_number': audio.tags.get('TRCK', 'Unknown Track Number'),
            'disc_number': audio.tags.get('TPOS', 'Unknown Disc Number'),
            'album_artist': audio.tags.get('TPE2', 'Unknown Album Artist'),
            'composer': audio.tags.get('TCOM', 'Unknown Composer'),
            'publisher': audio.tags.get('TPUB', 'Unknown Publisher'),
            'ISRC': audio.tags.get('TSRC', 'Unknown ISRC'),
            'catalog_number': audio.tags.get('TXXX:CATALOGNUMBER', 'Unknown Catalog Number'),
        }
        return metadata
    except Exception as e:
        logger.error(f"Error reading metadata for {track_path}: {e}")
        return None

# Example usage
get_track_metadata(tracks_paths[0])

2025-07-27 16:55:03,418 - __main__ - ERROR - Error reading metadata for D:\Soulseek Downloads\complete\(KARAT 04) Ark - Unknow Mysterioso\A1 - Taimz.flac: can't sync to MPEG frame


In [28]:
def update_text_frame(tag_dict, frame_id, value, frame_cls):
    """Update or add a text frame if value differs."""
    if frame_id not in tag_dict:
        tag_dict.add(frame_cls(encoding=3, text=value))
    elif tag_dict[frame_id].text[0] != value:
        tag_dict[frame_id] = frame_cls(encoding=3, text=value)
    
def update_txxx_frame(tag_dict, desc, value):
    existing = [frame for frame in tag_dict.getall("TXXX") if frame.desc == desc]
    if not existing or existing[0].text[0] != value:
        tag_dict.add(TXXX(encoding=3, desc=desc, text=value))


for track_path in tracks_paths:
    audio = MP3(track_path, ID3=ID3)
    
    if audio.tags is None:
        audio.add_tags()
    
    metadata = discogs_release_metadata
    
    update_text_frame(audio.tags, 'TALB', metadata['title'], TALB)
    update_text_frame(audio.tags, 'TCON', '; '.join(metadata['styles']), TCON)
    update_text_frame(audio.tags, 'TPUB', '; '.join(metadata['labels']), TPUB)
    update_txxx_frame(audio.tags, 'CATALOGNUMBER', '; '.join(metadata['catalog_numbers']))

    if 'TDRC' not in audio.tags:
        audio.tags.add(TDRC(encoding=3, text=str(metadata['year'])))  # Ensure year is str

    # Optional: Update artist/title if needed
    # update_text_frame(audio.tags, 'TPE1', metadata['artist'], TPE1)
    # update_text_frame(audio.tags, 'TIT2', metadata['title'], TIT2)
    
    audio.save()

In [ ]:
# Move files to the organized library
music_library_dest_dir = Path(f"D:/Music Collection/{discogs_release_metadata['artist'][0]}/{discogs_release_metadata['title']}")
music_library_dest_dir.mkdir(parents=True, exist_ok=True)

for item in tracks_paths + cover_art_paths:
    shutil.move(item, music_library_dest_dir)


## Select Release

This section demonstrates how to scan the download folder, tag audio files with Discogs metadata, move them to the organized library, and update the database.

In [ ]:
# from pathlib import Path
# from shutil import move
# from aft.db.database import SessionLocal, init_db
# from aft.db.models import Artist, Release, Track
# from aft.discogs_client import DiscogsClient
# from aft.credentials import discogs_user_token
# from aft.utils.normalization import normalize_spaces
# import logging

# # Initialize DB (run once)
# init_db()
# session = SessionLocal()

# download_dir = Path("D:/Soulseek Downloads/complete")
# music_library_dir = Path("D:/Music Collection")


In [ ]:
def organize_and_tag_downloads(download_dir, music_library_dir, discogs_client, session):
    """
    Scan download_dir for artist/release folders, tag audio files, move to music_library_dir, and update DB.
    """
    for artist_folder in download_dir.iterdir():
        if not artist_folder.is_dir():
            continue
        artist_name = normalize_spaces(artist_folder.name)
        # Get or create artist in DB
        artist = session.query(Artist).filter_by(name=artist_name).first()
        if not artist:
            artist = Artist(name=artist_name)
            session.add(artist)
            session.commit()
        for release_folder in artist_folder.iterdir():
            if not release_folder.is_dir():
                continue
            release_title = normalize_spaces(release_folder.name)
            # Search Discogs for release metadata
            releases = discogs_client.search_release(release_title, artist_name)
            if not releases:
                logger.warning(f"No Discogs release found for {release_title} by {artist_name}")
                continue
            release_meta = releases[0]
            # Get or create release in DB
            release = session.query(Release).filter_by(title=release_title, artist_id=artist.id).first()
            if not release:
                release = Release(
                    title=release_title,
                    year=release_meta.get('year'),
                    discogs_id=release_meta.get('id'),
                    artist=artist
                )
                session.add(release)
                session.commit()
            # Process tracks
            for track_file in release_folder.glob("*.mp3"):
                from mutagen.mp3 import MP3
                from mutagen.id3 import ID3, TIT2, TPE1, TALB, TDRC
                audio = MP3(track_file, ID3=ID3)
                # Update tags if missing
                if 'TIT2' not in audio:
                    # Try to match by filename or Discogs tracklist
                    track_title = track_file.stem
                    if release_meta.get('tracklist'):
                        # Try to match by position or filename
                        discogs_track = next((t for t in release_meta['tracklist'] if t['title'].lower() in track_title.lower()), None)
                        if discogs_track:
                            track_title = discogs_track['title']
                    audio['TIT2'] = TIT2(encoding=3, text=track_title)
                if 'TPE1' not in audio:
                    audio['TPE1'] = TPE1(encoding=3, text=artist_name)
                if 'TALB' not in audio:
                    audio['TALB'] = TALB(encoding=3, text=release_title)
                if 'TDRC' not in audio and release_meta.get('year'):
                    audio['TDRC'] = TDRC(encoding=3, text=str(release_meta['year']))
                audio.save()
                # Move file to organized library
                dest_dir = music_library_dir / artist_name / release_title
                dest_dir.mkdir(parents=True, exist_ok=True)
                dest_path = dest_dir / track_file.name
                move(str(track_file), str(dest_path))
                # Add track to DB
                track = session.query(Track).filter_by(title=track_title, release_id=release.id).first()
                if not track:
                    track = Track(
                        title=track_title,
                        position=None,
                        duration=None,
                        file_path=str(dest_path),
                        release=release
                    )
                    session.add(track)
                    session.commit()
    print("Organization and tagging complete.")

# Run the integration
organize_and_tag_downloads(download_dir, music_library_dir, discogs_client, session)

### Note add BPM and Key with librosa

In [ ]:

# Example usage
audio_file = 'your_audio_file.wav'  # Replace with your audio file path
bpm = get_bpm(audio_file)
print(f"Estimated BPM: {bpm:.2f}")

def get_key(audio_path):
    y, sr = librosa.load(audio_path)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)

    # Define major and minor key profiles (simplified example)
    major_profile = np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52, 5.19, 2.39, 3.66, 2.29, 2.88])
    minor_profile = np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54, 4.75, 3.98, 2.69, 3.34, 3.17])

    # Normalize profiles
    major_profile = major_profile / np.sum(major_profile)
    minor_profile = minor_profile / np.sum(minor_profile)

    # Calculate correlation for all 12 transpositions
    key_correlations = []
    for i in range(12):
        rotated_chroma = np.roll(chroma, i, axis=0) # Rotate chroma for each key
        major_corr = np.corrcoef(rotated_chroma.mean(axis=1), major_profile)[0, 1]
        minor_corr = np.corrcoef(rotated_chroma.mean(axis=1), minor_profile)[0, 1]
        key_correlations.append((major_corr, minor_corr))

    # Determine the most likely key
    key_names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
    best_correlation = -1
    estimated_key = "Unknown"

    for i, (major_corr, minor_corr) in enumerate(key_correlations):
        if major_corr > best_correlation:
            best_correlation = major_corr
            estimated_key = f"{key_names[i]} Major"
        if minor_corr > best_correlation:
            best_correlation = minor_corr
            estimated_key = f"{key_names[i]} Minor"
    
    return estimated_key

# Example usage
audio_file = 'your_audio_file.wav' # Replace with your audio file path
key = get_key(audio_file)
print(f"Estimated Key: {key}")

## Get release title and artist

In [18]:
def get_release_title_artist(metadata: Dict[str, Any]) -> Tuple[str, str]:
    """
    Get release title and artist from metadata.

    Parameters
    ----------
    metadata : Dict[str, Any]
        Metadata dictionary.

    Returns
    -------
    Tuple[str, str]
        Release title and artist.
    """
    release_title = normalize_spaces(metadata.get('album', 'Unknown Album').text[0])
    artist = normalize_spaces(metadata.get('artist', 'Unknown Artist').text[0])
    return release_title, artist

if metadata is not None:
    release_title, artist = get_release_title_artist(metadata=metadata)
    print(f"Release Title: {release_title}, Artist: {artist}")
else:
    print("No metadata available.")

Release Title: Satori EP, Artist: The Hacker


## Search for release

In [19]:
releases = discogs_client.search_release(release_title=release_title, artist=artist)
display(releases)

# Sample: Get a specific release by ID
sample_release = discogs_client.get_release_by_id(release_id=6946250)
display(sample_release)

[{'id': 3839860,
  'title': 'The Hacker - Satori EP',
  'artist': ['The Hacker'],
  'year': 2012,
  'genres': ['Electronic'],
  'styles': ['Techno', 'Electro', 'Tech House'],
  'labels': ['Correspondant'],
  'formats': ['Vinyl'],
  'catalog_numbers': ['Correspondant 09'],
  'country': 'Germany',
  'tracklist': [{'position': 'A1', 'title': 'Satori'},
   {'position': 'A2', 'title': 'White Funk (Offset Remix)'},
   {'position': 'B1', 'title': 'White Funk'},
   {'position': 'B2', 'title': 'White Funk (Daniel Maloso Remix)'}]},
 {'id': 6946250,
  'title': 'The Hacker - Satori EP',
  'artist': ['The Hacker'],
  'year': 2012,
  'genres': ['Electronic'],
  'styles': ['Techno', 'Electro', 'Tech House'],
  'labels': ['Correspondant'],
  'formats': ['File'],
  'catalog_numbers': ['CORRESPONDANT 09'],
  'country': 'Germany',
  'tracklist': [{'position': '1', 'title': 'Satori'},
   {'position': '2', 'title': 'White Funk (Offset Remix)'},
   {'position': '3', 'title': 'White Funk'},
   {'position': 

{'id': 6946250,
 'title': 'Satori EP',
 'artist': ['The Hacker'],
 'year': 2012,
 'genres': ['Electronic'],
 'styles': ['Techno', 'Electro', 'Tech House'],
 'labels': ['Correspondant'],
 'formats': ['File'],
 'catalog_numbers': ['CORRESPONDANT 09'],
 'country': 'Germany',
 'tracklist': [{'position': '1', 'title': 'Satori'},
  {'position': '2', 'title': 'White Funk (Offset Remix)'},
  {'position': '3', 'title': 'White Funk'},
  {'position': '4', 'title': 'White Funk (Daniel Maloso Remix)'}]}

In [ ]:
def search_release_metadata(metadata: Dict[str, Any], discogs_client: DiscogsClient) -> List[Dict[str, Any]]:
    """
    Search for a release on Discogs using metadata.

    Returns
    -------
    List[Dict[str, Any]]
        List of releases found on Discogs (empty if none found).
    """
    try:
        release_title, artist = get_release_title_artist(metadata=metadata)
        releases = discogs_client.search_release(release_title=release_title, artist=artist)
        if not releases:
            logger.warning(f"No matching releases found for {release_title} by {artist}")
            return []
        return releases
    except Exception as e:
        logger.error(f"Error processing release: {str(e)}")
        return []

def select_releases_by_format(releases: List[Dict[str, Any]], format_type: str) -> Optional[Dict[str, Any]]:
    """
    Select the first release matching the format_type, or the first release if none match.

    Returns
    -------
    Optional[Dict[str, Any]]
        The selected release, or None if releases is empty.
    """
    if not releases:
        logger.warning("No releases provided to select from.")
        return None
    filtered = [release for release in releases if format_type in release.get("formats", [])]
    if not filtered:
        logger.info(f"No releases found with format {format_type}. Selecting the first release.")
        return releases[0]
    if len(filtered) > 1:
        logger.info(f"Multiple releases found with format {format_type}. Selecting the first one.")
        return filtered[0]
    return filtered[0]

# Search for release
if metadata is not None:
    releases = search_release_metadata(metadata=metadata, discogs_client=discogs_client)
    if not releases:
        logger.error("No releases found. Aborting.")
    elif len(releases) == 1:
        logger.info(f"Single release found for {release_title} by {artist}")
        release = releases[0]
        display(release)
    else:
        format_type = 'File'
        logger.info(f"Multiple releases found for {release_title} by {artist}. Selecting by format: {format_type}")
        
        # Get differnt formats
        # formats = set()
        # for release in releases:
        #     for fmt in release["formats"]:
        #         formats.add(fmt.get("name", "Unknown Format"))
        # display(formats)
        
        release = select_releases_by_format(releases, format_type)
        if release:
            display(release)
        else:
            logger.error("No suitable release found after filtering.")


2025-06-27 21:44:13,195 - __main__ - INFO - Multiple releases found for Satori EP by The Hacker. Selecting by format: File


AttributeError: 'str' object has no attribute 'get'

## Exmample

In [8]:
music_library_dir = Path("D:/Music Collection")
artist_name = r"9 Lazy 9"

# Join the paths in a cross-platform way
full_path = music_library_dir / artist_name

# List directories inside the path
if full_path.exists() and full_path.is_dir():
    folders = [f.name for f in full_path.iterdir() if f.is_dir()]
    print("Folders inside:", full_path)
    print(folders)
else:
    print("Path does not exist or is not a directory.")


Folders inside: D:\Music Collection\9 Lazy 9
['Electric Lazyland', 'Paradise Blown', 'Sweet Jones']


In [9]:
# Instantiate a client
discogs_client = discogs_client.Client(
    'discogs_api_example/1.0',
    user_token=discogs_user_token
)

In [10]:
release_name = folders[0]

# Search for a release
search_results = discogs_client.search(
    release_name,
    type='release',
    artist=artist_name
)


# releases = search_results

# for release in releases:
#     # if release.title == release_name:
#     print(release)

# release = search_results[0]
# styles = release.styles


In [11]:
styles = []
labels = []
catalog_numbers = []
countries = []

for release in search_results:
    if release_name in release.title:
        # Get styles     
        styles.extend(release.styles)
        
        # Get labels and catalog numbers
        for label in release.labels:
            labels.append(label.name)
            catalog_numbers.append(label.catno)  # Get catalog number
        
        # Get country
        countries.append(release.country)

# Print results
print("Release:", release_name)
print("Styles:", list(set(styles)))
print("Labels:", list(set(labels)))
print("Catalog Numbers:", list(set(catalog_numbers)))
print("Countries:", list(set(countries)))  # Print country


Release: Electric Lazyland
Styles: ['Acid Jazz', 'Big Beat', 'Trip Hop', 'Downtempo']
Labels: ['Ninja Tune']
Catalog Numbers: ['ZEN CD 14', 'ZEN14', 'zen cd 14', 'ZEN 1238', 'ZEN 14', 'zen 14', 'ZENCD 14']
Countries: ['Canada', 'UK']


In [29]:
# Fetch release by ID (replace with actual release ID)
release_id = 249504  # Example: Pink Floyd - The Dark Side of the Moon
release = discogs_client.release(release_id)

# Print some attributes
print(f"Title: {release.title}")
print(f"Artist(s): {[artist.name for artist in release.artists]}")
print(f"Genres: {release.genres}")
print(f"Year: {release.year}")
print(f"Labels: {[label.name for label in release.labels]}")
print(f"Tracklist:")
for track in release.tracklist:
    print(f" - {track.position}: {track.title} ({track.duration})")


Title: Never Gonna Give You Up
Artist(s): ['Rick Astley']
Genres: ['Electronic', 'Pop']
Year: 1987
Labels: ['RCA']
Tracklist:
 - A: Never Gonna Give You Up (3:32)
 - B: Never Gonna Give You Up (Instrumental) (3:30)


In [3]:
from tagging.discogs_client import DiscogsClient
from tagging.credentials import discogs_user_token

client = DiscogsClient(discogs_user_token)



## Search for a release

In [43]:
search_results = client.search_release(title='Electric Lazyland', artist='9 Lazy 9')

# Print search results
for release in search_results:
    print(release)

In [45]:
release_id = 249504
release = client.get_release_by_id(release_id)
print(release)


{'id': 249504, 'title': 'Never Gonna Give You Up', 'artist': ['Rick Astley'], 'year': 1987, 'genres': ['Electronic', 'Pop'], 'styles': ['Euro-Disco'], 'labels': ['RCA'], 'catalog_numbers': ['PB 41447'], 'country': 'UK'}
